# Bangladesh Global Study Abroad Cost & Scholarship Dataset (2027) — Starter EDA

**Publisher:** [EduExpress International](https://eduexpressint.com/press) (Dhanmondi, Dhaka, Bangladesh)  
**License:** Creative Commons Attribution 4.0 International (`CC BY 4.0`)  
**Release:** `v2026.11.26` · **Zenodo Version DOI:** [10.5281/zenodo.24545329](https://doi.org/10.5281/zenodo.24545329) · **Concept DOI:** [10.5281/zenodo.24423800](https://doi.org/10.5281/zenodo.24423800)  
**Interactive 32-Country Cost & Scholarship Calculator:** [https://eduexpressint.com/compare-costs](https://eduexpressint.com/compare-costs)

---

### What This Starter Notebook Explores
1. **32-Country Global Bangladesh Outbound Benchmark:** Comparing 12 active EduExpress destinations alongside 20 top OECD/regional destinations across Asia, Europe, North America, Oceania, and the Middle East with late-2026 statutory visa solvency thresholds.
2. **Expanding Multi-Country Verified University Catalog (`362` Active):** Currently **353 China universities** across **93 cities** (including **306 / 306 CSC-designated universities**) + **9 Malaysia universities**, with UK, Hungary, South Korea, Finland, and European institutional rosters expanding in upcoming versions.
3. **Explicit Single-Year Tuition Quartiles & Separated Non-Tuition Fees:** Analyzing **1,796 programs** while keeping institutional tuition strictly separate from dormitory, insurance, and arrival fees (**1,497 itemized fee rows**).
4. **Structured Scholarship Accounting (`736` Total):** Breakdown of **682 university scholarship tracks** (zero vague entries) plus **54 Bangladesh-eligible flagship national scholarships** across all 32 destinations.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Brand-aligned editorial palette (EduExpress clear desk)
PETROL = "#0c3733"
TEAL = "#0f766e"
JADE = "#1b7a57"
MARIGOLD = "#e9a21b"
CLAY = "#b4462f"
SLATE = "#475569"

from IPython.display import display

# Auto-detect Kaggle input directory (handles both legacy /kaggle/input/<slug> and 2026 /kaggle/input/datasets/<user>/<slug>) or local path
def locate_data_dir():
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        found = next(kaggle_root.rglob("universities_verified.csv"), None)
        if found is not None:
            return found.parent
    for candidate in (Path("../data"), Path("data"), Path(".")):
        if (candidate / "universities_verified.csv").exists():
            return candidate
    return None

DATA_DIR = locate_data_dir()
assert DATA_DIR is not None, "Could not locate universities_verified.csv under /kaggle/input or local data/"
print(f"Loaded dataset from: {DATA_DIR}")

unis = pd.read_csv(DATA_DIR / "universities_verified.csv")
programs = pd.read_csv(DATA_DIR / "programs_verified.csv")
scholarships = pd.read_csv(DATA_DIR / "scholarships_verified.csv")
fees = pd.read_csv(DATA_DIR / "itemized_fees_verified.csv")
dest32 = pd.read_csv(DATA_DIR / "destinations_matrix_32_countries.csv")
country_schol = pd.read_csv(DATA_DIR / "country_scholarships_verified.csv")

summary_df = pd.DataFrame([
    {"Table": "universities_verified.csv", "Rows": len(unis), "Key Highlight": f"{(unis['country']=='China').sum()} China ({(unis['is_csc_designated']=='yes').sum()}/306 CSC) + {(unis['country']=='Malaysia').sum()} Malaysia"},
    {"Table": "programs_verified.csv", "Rows": len(programs), "Key Highlight": f"{programs['level'].nunique()} degree/language levels across {programs['city'].nunique()} cities"},
    {"Table": "scholarships_verified.csv", "Rows": len(scholarships), "Key Highlight": f"{(scholarships['waives_accommodation']=='yes').sum()} include dorm waiver · {scholarships['monthly_stipend_cny'].notna().sum()} list explicit CNY stipend"},
    {"Table": "itemized_fees_verified.csv", "Rows": len(fees), "Key Highlight": "Non-tuition dorm, insurance, residence permit, EMGS & arrival fees kept strictly separate"},
    {"Table": "destinations_matrix_32_countries.csv", "Rows": len(dest32), "Key Highlight": f"{(dest32['service_status']!='Benchmark Reference').sum()} Active EduExpress + {(dest32['service_status']=='Benchmark Reference').sum()} Global Benchmark Reference"},
    {"Table": "country_scholarships_verified.csv", "Rows": len(country_schol), "Key Highlight": "Bangladesh-eligible government, regional & institutional flagship scholarships"},
])
summary_df


## 1. China University Footprint & Complete `306 / 306` CSC Coverage
Let's inspect the geographic footprint of China's verified catalog across **93 cities**, comparing CSC-designated vs. non-CSC universities and verifying English-taught and Chinese Language program availability.


In [ ]:
china_unis = unis[unis["country"] == "China"].copy()
print(f"Total active China universities: {len(china_unis)}")
print(f"CSC-designated China universities: {(china_unis['is_csc_designated'] == 'yes').sum()} / 306")
print(f"Offering English-taught programs: {(china_unis['has_english_taught'] == 'yes').sum()}")
print(f"Offering Non-Degree Chinese Language programs: {(china_unis['has_chinese_language_program'] == 'yes').sum()}")
print(f"Distinct Chinese cities covered: {china_unis['city'].nunique()}")

city_counts = (
    china_unis.groupby(["city", "is_csc_designated"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .sort_values("total", ascending=False)
    .head(15)
    .iloc[::-1]
)

fig, ax = plt.subplots(figsize=(10, 6))
csc_vals = city_counts.get("yes", pd.Series(0, index=city_counts.index))
non_csc_vals = city_counts.get("no", pd.Series(0, index=city_counts.index))

ax.barh(city_counts.index, csc_vals, color=PETROL, label="CSC-Designated (306/306 roster)")
ax.barh(city_counts.index, non_csc_vals, left=csc_vals, color=MARIGOLD, label="Additional Verified China Universities")

for idx, (city, row) in enumerate(city_counts.iterrows()):
    ax.text(row["total"] + 0.6, idx, str(int(row["total"])), va="center", fontsize=9, fontweight="bold", color=PETROL)

ax.set_title("Top 15 Chinese Cities by Verified University Coverage (2027 Benchmark)", fontsize=13, fontweight="bold", color=PETROL, pad=14)
ax.set_xlabel("Verified Active Universities")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()


## 2. Explicit Single-Year China Tuition Distribution (CNY)
Following EduExpress International's **Evidence-First methodology**, we extract strictly **explicit single-year CNY tuition rows** (excluding ranged estimates, semester-only non-degree tracks, and `"Confirm with university"` rows) so quartiles reflect verifiable institutional schedules rather than blended marketing averages.


In [ ]:
def parse_single_year_cny(val):
    if pd.isna(val):
        return np.nan
    text = str(val).strip()
    if not text or re.search(r"semester|month|total|confirm|contact|varied|tbd", text, re.I):
        return np.nan
    if not re.search(r"CNY|RMB|¥", text, re.I):
        return np.nan
    nums = [int(n.replace(",", "")) for n in re.findall(r"\d{1,3}(?:,\d{3})+|\d{4,6}", text)]
    # Strict single-year explicit figure only (exclude ranges like 18,000 - 24,000)
    if len(nums) != 1:
        return np.nan
    val_cny = nums[0]
    return val_cny if 3000 <= val_cny <= 400000 else np.nan

china_progs = programs[programs["country"] == "China"].copy()
china_progs["tuition_cny"] = china_progs["tuition"].apply(parse_single_year_cny)
explicit = china_progs.dropna(subset=["tuition_cny"]).copy()

def classify_track(row):
    lvl = str(row["level"]).lower()
    lang = str(row["languages"]).lower()
    if "bachelor" in lvl and "english" in lang:
        return "English Bachelor"
    if "bachelor" in lvl and "chinese" in lang:
        return "Chinese Bachelor"
    if "master" in lvl:
        return "Master's"
    if "phd" in lvl or "doctoral" in lvl:
        return "PhD"
    return "Other / Language / Diploma"

explicit["track"] = explicit.apply(classify_track, axis=1)
track_order = ["Chinese Bachelor", "English Bachelor", "Master's", "PhD"]
quartile_rows = []
for tr in track_order:
    s = explicit.loc[explicit["track"] == tr, "tuition_cny"]
    if len(s):
        quartile_rows.append({
            "Track": tr,
            "Explicit Rows": len(s),
            "Min (CNY)": int(s.min()),
            "Q1 25th (CNY)": int(s.quantile(0.25)),
            "Median (CNY)": int(s.median()),
            "Q3 75th (CNY)": int(s.quantile(0.75)),
            "Max (CNY)": int(s.max()),
        })
quartiles_df = pd.DataFrame(quartile_rows)
print(f"Total explicit single-year China CNY rows analyzed: {len(explicit)} (Overall Median: CNY {int(explicit['tuition_cny'].median()):,}/yr)")
display(quartiles_df)

# Boxplot (capped at CNY 80,000 visual window to keep core quartiles readable; outliers noted)
fig, ax = plt.subplots(figsize=(10, 5.2))
plot_data = [explicit.loc[(explicit["track"] == tr) & (explicit["tuition_cny"] <= 80000), "tuition_cny"].values for tr in track_order]
bp = ax.boxplot(plot_data, patch_artist=True, widths=0.52, medianprops=dict(color=MARIGOLD, linewidth=2.5))
ax.set_xticks(range(1, len(track_order) + 1))
ax.set_xticklabels(track_order)
colors = [TEAL, PETROL, JADE, SLATE]
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.85)

for i, tr in enumerate(track_order, start=1):
    med = explicit.loc[explicit["track"] == tr, "tuition_cny"].median()
    ax.text(i, med + 1400, f"Median: CNY {int(med):,}", ha="center", fontsize=8.5, fontweight="bold", color=PETROL)

ax.set_title("China Explicit Annual Tuition Distribution by Track (CNY/year, ≤ CNY 80k core window)", fontsize=12.5, fontweight="bold", color=PETROL, pad=12)
ax.set_ylabel("Annual Tuition (CNY)")
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()


## 3. Scholarship Structure & Waiver Coverage (`682` University Tracks + `54` Flagship National)
Gross published tuition rarely equals what a qualified Bangladeshi applicant pays in China. Every university scholarship record separates **tuition waivers**, **accommodation/insurance waivers**, and **monthly living stipends** (never deducted from upfront arrival cash fees).


In [ ]:
waiver_counts = scholarships["tuition_waiver_type"].fillna("unspecified").value_counts()
dorm_waiver_pct = (scholarships["waives_accommodation"] == "yes").mean() * 100
ins_waiver_pct = (scholarships["waives_insurance"] == "yes").mean() * 100
scholarships["monthly_stipend_cny"] = pd.to_numeric(scholarships["monthly_stipend_cny"], errors="coerce")
stipend_rows = scholarships.dropna(subset=["monthly_stipend_cny"])

print(f"Total university scholarship tracks: {len(scholarships)}")
print(f"Include on-campus accommodation waiver: {(scholarships['waives_accommodation']=='yes').sum()} ({dorm_waiver_pct:.1f}%)")
print(f"Include CNY 800/yr medical insurance waiver: {(scholarships['waives_insurance']=='yes').sum()} ({ins_waiver_pct:.1f}%)")
print(f"List explicit monthly living stipend (CNY): {len(stipend_rows)} (Median: CNY {int(stipend_rows['monthly_stipend_cny'].median()):,}/month)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

waiver_labels = {
    "percent": "Percentage Tuition Waiver",
    "varies": "Multi-Tier / Merit-Based",
    "fixed_cny": "Fixed CNY Tuition Deduction",
    "net_tuition_cny": "Fixed Net Tuition Schedule",
    "none": "Stipend / Non-Tuition Only",
}
w_series = waiver_counts.rename(index=waiver_labels)
bar_colors = [CLAY, SLATE, MARIGOLD, TEAL, PETROL][-len(w_series):]
ax1.barh(w_series.index[::-1], w_series.values[::-1], color=bar_colors)
for idx, val in enumerate(w_series.values[::-1]):
    ax1.text(val + 6, idx, str(int(val)), va="center", fontsize=9, fontweight="bold", color=PETROL)
ax1.set_title("University Scholarships by Waiver Type (n = 682)", fontsize=11.5, fontweight="bold", color=PETROL)
ax1.set_xlabel("Scholarship Records")

cat_counts = country_schol["scholarship_category"].value_counts()
ax2.barh(cat_counts.index[::-1], cat_counts.values[::-1], color=JADE)
for idx, val in enumerate(cat_counts.values[::-1]):
    ax2.text(val + 0.4, idx, str(int(val)), va="center", fontsize=9, fontweight="bold", color=PETROL)
ax2.set_title("32-Country Flagship Scholarships by Category (n = 54)", fontsize=11.5, fontweight="bold", color=PETROL)
ax2.set_xlabel("Bangladesh-Eligible National Programs")

plt.tight_layout()
plt.show()


## 4. 32-Country Global Bangladesh Outbound Matrix (2026/27 Visa & Cost Benchmark)
Beyond China and Malaysia, the dataset covers **32 study destinations** (`12` Active EduExpress destinations + `20` Global Benchmark Reference destinations) with late-2026 statutory visa solvency thresholds (Canada CAD 23,448, Germany EUR 11,904 Sperrkonto, France EUR 10,530 Decree 2026-526, Australia AUD 29,710, Finland EUR 9,600, Netherlands EUR 13,569, Sweden SEK 127,872, Norway NOK 170,368, etc.).


In [ ]:
region_summary = (
    dest32.groupby(["region", "service_status"])
    .size()
    .unstack(fill_value=0)
    .assign(Total=lambda d: d.sum(axis=1))
    .sort_values("Total", ascending=False)
)
display(region_summary)

# Show clean comparison table of key 2026/27 statutory visa financial proof thresholds
cols = ["country", "region", "service_status", "currency", "indicative_tuition", "statutory_visa_financial_proof"]
dest32[cols].head(20)


## 5. Citation & Further Exploration
If you use this dataset or notebook in higher-education journalism, policy analysis, or machine learning projects, please cite:

> EduExpress International & Abdullah Al Rakib (2026). *Bangladesh Study Abroad, Scholarship & China University Cost Benchmark Dataset (2027)* (Version `2026.11.26`) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.24545329 (Concept DOI: https://doi.org/10.5281/zenodo.24423800)

- **Interactive Calculator & Embed Widget:** https://eduexpressint.com/compare-costs · https://eduexpressint.com/embed/cost-calculator
- **Press Room & Live JSON APIs:** https://eduexpressint.com/press
- **GitHub Source Repository:** https://github.com/rakibnuist/bangladesh-study-abroad-dataset
